In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from scorecard.binning import bin_categorical, bin_continuous
from scorecard.data import load_german_credit
from scorecard.inference import score_new_applicant
from scorecard.scorecard import build_scorecard, score_applicant
from scorecard.woe import apply_woe, compute_woe_iv

pd.set_option("display.max_columns", None)

SEED = 42

In [ ]:
df = load_german_credit("../data/raw/german_credit/german.data")
df.head()

In [ ]:
categorical_cols = df.select_dtypes(include=["str"]).columns.tolist()
continuous_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
continuous_cols.remove("target")

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(y_train.mean(), y_test.mean())

In [ ]:
iv_results = {}
woe_tables = {}
train_binned_cols = {}
test_binned_cols = {}

for col in categorical_cols:
    train_b, test_b, rare = bin_categorical(X_train[col], X_test[col])
    summary, iv = compute_woe_iv(train_b, y_train)
    train_binned_cols[col] = train_b
    test_binned_cols[col] = test_b
    woe_tables[col] = summary
    iv_results[col] = iv

for col in continuous_cols:
    train_b, test_b, edges = bin_continuous(X_train[col], X_test[col])
    summary, iv = compute_woe_iv(train_b, y_train)
    train_binned_cols[col] = train_b
    test_binned_cols[col] = test_b
    woe_tables[col] = summary
    iv_results[col] = iv

iv_ranking = pd.Series(iv_results).sort_values(ascending=False)
iv_ranking

In [ ]:
keep_features = iv_ranking[iv_ranking > 0.02].index.tolist()  # type: ignore
print(f"Keeping {len(keep_features)} of {len(iv_ranking)} features with IV > 0.02")
print(keep_features)

In [ ]:
X_train_woe_full = pd.DataFrame(index=X_train.index)
X_test_woe_full = pd.DataFrame(index=X_test.index)

for col in keep_features:
    X_train_woe_full[col] = apply_woe(train_binned_cols[col], woe_tables[col])
    X_test_woe_full[col] = apply_woe(test_binned_cols[col], woe_tables[col])

In [ ]:
for col in keep_features:
    n_nan = X_test_woe_full[col].isna().sum()
    if n_nan > 0:
        print(f"{col}: {n_nan} NaNs")

In [ ]:
corr_matrix = X_train_woe_full[keep_features].corr()
print(corr_matrix)

In [ ]:
model_full = LogisticRegression(random_state=SEED)
model_full.fit(X_train_woe_full[keep_features], y_train)

In [ ]:
p_test_full = model_full.predict_proba(X_test_woe_full[keep_features])[:, 1]
auc_full = roc_auc_score(y_test, p_test_full)
gini_full = 2 * auc_full - 1
print(f"AUC: {auc_full:.4f}, Gini: {gini_full:.4f}")

In [ ]:
pd.Series(model_full.coef_[0], index=keep_features).sort_values()

In [ ]:
print(f"Mean predicted PD: {p_test_full.mean():.4f}")
print(f"Actual bad rate (test): {y_test.mean():.4f}")

In [ ]:
from scipy.stats import ks_2samp

ks_stat, ks_pval = ks_2samp(p_test_full[y_test == 1], p_test_full[y_test == 0])
print(f"KS statistic: {ks_stat:.4f}")

In [ ]:
OFFSET = 600
PDO = 20
FACTOR = PDO / np.log(2)

In [ ]:
beta_0 = model_full.intercept_[0]
beta_checking = pd.Series(model_full.coef_[0], index=keep_features)["checking_status"]
woe_checking_A14 = woe_tables["checking_status"].loc["A14", "woe"]

In [ ]:
beta_0

In [ ]:
print(woe_checking_A14)
print(beta_checking)

In [ ]:
n = len(keep_features)
points_A14 = (
    -beta_checking * woe_checking_A14 * FACTOR - (beta_0 * FACTOR / n) + (OFFSET / n)
)
print(points_A14)

In [ ]:
woe_checking_A11 = woe_tables["checking_status"].loc["A11", "woe"]
points_A11 = (
    -beta_checking * woe_checking_A11 * FACTOR - (beta_0 * FACTOR / n) + (OFFSET / n)
)
print(points_A11)

In [ ]:
print(woe_checking_A11)

In [ ]:
scorecard_table = build_scorecard(
    model_full, woe_tables, keep_features, offset=OFFSET, pdo=PDO
)
scorecard_table.sort_values(["feature", "points"])

## Worked example — scoring a single applicant

In [ ]:
applicant_idx = 0
applicant_row = X_test.iloc[applicant_idx]
print(applicant_row)

In [ ]:
applicant_binned = {
    col: test_binned_cols[col].iloc[applicant_idx] for col in keep_features
}
applicant_score = score_applicant(applicant_binned, scorecard_table)
print(f"\nTotal scorecard points: {applicant_score:.2f}")

## Persist the fitted artifacts

In [ ]:
joblib.dump(model_full, "../artifacts/model.pkl")
joblib.dump(woe_tables, "../artifacts/woe_tables.pkl")
joblib.dump(scorecard_table, "../artifacts/scorecard_table.pkl")
joblib.dump(keep_features, "../artifacts/keep_features")

binning_artifacts = {}

for col in categorical_cols:
    if col not in keep_features:
        continue


joblib.dump(binning_artifacts, "../artifacts/binning_artifacts.pkl")

In [ ]:
for col in categorical_cols:
    if col not in keep_features:
        continue
    train_b, test_b, rare = bin_categorical(X_train[col], X_test[col])
    binning_artifacts[col] = {
        "type": "categorical",
        "rare_levels": rare,
        "known_levels": X_train[col].unique().tolist(),
    }

for col in continuous_cols:
    if col not in keep_features:
        continue
    train_b, test_b, edges = bin_continuous(X_train[col], X_test[col])
    binning_artifacts[col] = {
        "type": "continuous",
        "edges": edges,
    }

print(f"Captured binning artifacts for {len(binning_artifacts)} features")
print(binning_artifacts.keys())

In [ ]:
joblib.dump(model_full, "../artifacts/model.pkl")
joblib.dump(woe_tables, "../artifacts/woe_tables.pkl")
joblib.dump(scorecard_table, "../artifacts/scorecard_table.pkl")
joblib.dump(keep_features, "../artifacts/keep_features.pkl")
joblib.dump(binning_artifacts, "../artifacts/binning_artifacts.pkl")

print("Artifacts saved.")

In [ ]:
print(binning_artifacts)

In [ ]:
sorted_by_pd = np.argsort(
    model_full.predict_proba(X_test_woe_full[keep_features])[:, 1]
)
low_risk_idx = sorted_by_pd[0]
high_risk_idx = sorted_by_pd[-1]

In [ ]:
raw_applicant = X_test.iloc[low_risk_idx][keep_features].to_dict()
result = score_new_applicant(
    raw_applicant,
    binning_artifacts,
    woe_tables,
    scorecard_table,
    model_full,
    keep_features,
)
print(result)